#Data Clean

In [ ]:
!pip install lingua-language-detector
!pip install deep_translator
!pip install vaderSentiment
!pip install xgboost

import pandas            as pd
import numpy             as np
import matplotlib.pyplot as plt
import seaborn           as sns
import xgboost           as xgb
import graphviz
import re
import io
import os

from sklearn                         import datasets, tree
from sklearn.compose                 import ColumnTransformer
from sklearn.ensemble                import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.inspection              import PartialDependenceDisplay
from sklearn.linear_model            import LinearRegression, Ridge, LogisticRegression
from sklearn.model_selection         import (
  cross_val_score,
  cross_validate,
  GridSearchCV,
  RandomizedSearchCV,
  StratifiedKFold,
  train_test_split
)

from sklearn.metrics                 import (
  accuracy_score,
  classification_report,
  confusion_matrix,
  ConfusionMatrixDisplay,
  f1_score,
  mean_absolute_error,
  mean_squared_error,
  precision_score,
  r2_score,
  recall_score,
  roc_auc_score
)

from sklearn.pipeline                import Pipeline
from sklearn.preprocessing           import OneHotEncoder, StandardScaler
from sklearn.tree                    import DecisionTreeRegressor, DecisionTreeClassifier
from IPython.display                 import display
from xgboost                         import XGBRegressor, XGBClassifier

from deep_translator                 import GoogleTranslator
from lingua                          import Language, LanguageDetectorBuilder
from textblob                        import TextBlob
from vaderSentiment.vaderSentiment   import SentimentIntensityAnalyzer

from google.colab import files, drive, userdata

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/ddds-cohort-21/Projects/Capstone/Data/chat_metrics_20260730_062950.xlsx'

dfs = pd.read_excel(file_path, sheet_name=None)
df = dfs['Raw Data']
feedback = dfs['User Feedback']

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 1.2 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
#Raw Data
#Drop excess columns
df.drop(columns=['user_id', 'correlation_id', 'hour', 'date', 'secondary_category'], inplace=True)

#Convert from UTC to Mountain Time (Albuquerque)
df['timestamp'] = df['timestamp'].dt.tz_localize('UTC').dt.tz_convert('America/Denver')

#Update day of week to match local time
df['day_of_week'] = df['timestamp'].dt.day_name()

#Fill primary category nulls
df.loc[814, 'primary_category']= 'sunport_amenities'
df.loc[816, 'primary_category']= 'navigation'
df.loc[817, 'primary_category']= 'navigation'
df.loc[818, 'primary_category']= 'airline_logistics'

#Feedback
#Drop null star rating
feedback.drop([147, 262, 309, 311, 761], inplace=True)

#Drop unneccesary columns
feedback.drop(columns=['timestamp', 'ip_address', 'feedback_text'], inplace=True)

#Drop Duplicates
feedback = feedback.drop_duplicates()

#Merge
df = df.merge(feedback[['chat_id', 'star_rating']], on='chat_id', how='left')

#Emojis
emoji_pattern = re.compile(
    '['
    '\U0001F600-\U0001F64F'  # emoticons
    '\U0001F300-\U0001F5FF'  # symbols & pictographs
    '\U0001F680-\U0001F6FF'  # transport & map symbols
    '\U0001F1E0-\U0001F1FF'  # flags
    ']+',
    flags=re.UNICODE
)

#Find questions with emojis
emoji_questions = df[df['question'].astype(str).apply(lambda x: bool(emoji_pattern.search(x)))]

#Replace emojies with Okay
df['question'] = df['question'].replace('👍', 'Okay')

#Find different languages
detector = (LanguageDetectorBuilder.from_all_languages().build())

def detect_language(text, threshold=0.55):
  if not isinstance(text, str) or text.strip() == '':
    return 'ENGLISH'

  #Get confidence values for all languages
  confidence_values = detector.compute_language_confidence_values(text)

  #Check if highest matched meets your threshold
  if confidence_values and confidence_values[0].value > threshold:
    return confidence_values[0].language.name #Added name here

  return 'ENGLISH'

df['question_language'] = df['question'].apply(detect_language)
df['answer_language'] = df['answer'].apply(detect_language)

#Replace false positives
df['answer_language'] = df['answer_language'].replace(['LATIN', 'YORUBA', 'ESPERANTO'], 'ENGLISH')

#Translate
translator = GoogleTranslator(source='auto', target='en')

def translate_to_english(text):
  if not isinstance(text, str) or text.strip() == '':
    return text

  try:
    return translator.translate(text)
  except Exception:
    return text

#Translate questions
df['question_en'] = df['question']

mask = df['question_language'] != 'ENGLISH'

df.loc[mask, 'question_en'] = (df.loc[mask, 'question'].apply(translate_to_english))

#Translate answers
df['answer_en'] = df['answer']

mask = df['answer_language'] != 'ENGLISH'

df.loc[mask, 'answer_en'] = (df.loc[mask, 'answer'].apply(translate_to_english))

#Clean up and drop
df.drop(columns=['chat_id', 'question', 'answer', 'question_language', 'answer_language'], inplace=True)

In [ ]:
#Aggregate
def round_to_half(x):
  return round(x * 2) / 2

df_clean = df.groupby('session_id').agg(
    timestamp = ('timestamp', 'first'),
    day_of_week = ('day_of_week', lambda x: x.mode().loc[0] if not x.mode().empty else None),
    processing_time_seconds = ('processing_time_seconds', 'sum'),
    total_tokens = ('total_tokens', 'sum'),
    input_tokens = ('input_tokens', 'sum'),
    output_tokens = ('output_tokens', 'sum'),
    model_calls = ('model_calls', 'sum'),
    tool_calls_count = ('tool_calls_count', 'sum'),
    step_summaries_count = ('step_summaries_count', 'sum'),
    question_length = ('question_length', 'sum'),
    answer_length = ('answer_length', 'sum'),
    primary_category = ('primary_category', lambda x: ', '.join(x.dropna().unique())),
    selected_agent = ('selected_agent', lambda x: ', '.join(x.dropna().unique())),
    has_geolocation = ('has_geolocation', 'any'),
    star_rating = ('star_rating', lambda x: round_to_half(x.mean()) if pd.notna(x.mean()) else np.nan),
    question_en = ('question_en', lambda x: ' '.join(x.dropna())),
    answer_en = ('answer_en', lambda x: ' '.join(x.dropna()))
).reset_index()

#Drop identifier
# df_clean.drop(columns=['session_id', 'step_summaries_count'], inplace=True)

#Keep sessions with star_rating only
df_clean.dropna(subset=['star_rating'], inplace=True)

#Binary Satisfaction
df_clean['satisfaction'] = (df_clean['star_rating'] >= 4).astype(int)

df_clean['satisfaction'].value_counts(normalize=True)

#Set target
target = 'satisfaction'

#Multi-Label Binary Encoding
#Primary Categories
categories = [
  'greetings',
  'sunport_amenities',
  'navigation',
  'airline_logistics',
  'general_info'
]

for category in categories:
  df_clean[f'primary_category_{category}'] = (
    df_clean['primary_category']
    .str.contains(category, regex=False, na=False)
    .astype('int8')
  )

df_clean.drop(columns='primary_category', inplace=True)

#Selected Agent
agents = [
  'reporter',
  'planner',
  'location',
  'location_fallback_to_reporter',
  'broad_search_synthesis',
  'broad_search_passthrough'
]

for agent in agents:
  df_clean[f'selected_agent{agent}'] = (
    df_clean['selected_agent']
    .str.contains(agent, regex=False, na=False)
    .astype('int8')
  )

df_clean.drop(columns='selected_agent', inplace=True)

#Datetime
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])

df_clean['year'] = df_clean['timestamp'].dt.year
df_clean['month'] = df_clean['timestamp'].dt.month
df_clean['day'] = df_clean['timestamp'].dt.day
df_clean['hour'] = df_clean['timestamp'].dt.hour

df_clean.drop(columns='timestamp', inplace=True)

#Cyclical Encoding
#Days of the Week
days = {
  'Monday':0,
  'Tuesday':1,
  'Wednesday':2,
  'Thursday':3,
  'Friday':4,
  'Saturday':5,
  'Sunday':6
}

df_clean['day_num'] = df_clean['day_of_week'].map(days)

df_clean['day_sin'] = np.sin(2 * np.pi * df_clean['day_num'] / 7)
df_clean['day_cos'] = np.cos(2 * np.pi * df_clean['day_num'] / 7)

#Hour
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

#Month
df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12)
df_clean['month_cos'] = np.sin(2 * np.pi * df_clean['month'] / 12)

#Term Frequency-Inverse Document Frequency
#Question TF-IDF
tfidf_question = TfidfVectorizer(max_features=500, stop_words='english')

question_tfidf = tfidf_question.fit_transform(df_clean['question_en'].fillna(''))

question_features = tfidf_question.get_feature_names_out()

#Weight of each word across all questions
scores = np.asarray(question_tfidf.sum(axis=0)).flatten()

top_words = pd.DataFrame({'word': question_features, 'tfidf_score':scores}).sort_values('tfidf_score', ascending=False)

#Answer TF-IDF
tfidf_answer = TfidfVectorizer(max_features=500, stop_words='english')

answer_tfidf = tfidf_answer.fit_transform(df_clean['answer_en'].fillna(''))

answer_features = tfidf_answer.get_feature_names_out()

#Weight of each word across all questions
scores = np.asarray(answer_tfidf.sum(axis=0)).flatten()

top_words = pd.DataFrame({'word': answer_features, 'tfidf_score':scores}).sort_values('tfidf_score', ascending=False)

#Sentiment Analysis
#Create analyzer
analyzer = SentimentIntensityAnalyzer()

#Create function
def get_sentiment(text):
  if pd.isna(text) or str(text).strip() == '':
    return 0

  return analyzer.polarity_scores(str(text))['compound']

#Question Sentiment
df_clean['question_sentiment'] = df_clean['question_en'].apply(get_sentiment)

#Answer Sentiment
df_clean['answer_sentiment'] = df_clean['answer_en'].apply(get_sentiment)

#Create sentiment score categories
def sentiment_category(score):
  if score >= 0.75:
    return 'positive'
  elif score <= 0.25:
    return 'negative'
  else:
    return 'neutral'

df_clean['question_sentiment_category'] = (df_clean['question_sentiment'].apply(sentiment_category))
df_clean['answer_sentiment_category'] = (df_clean['answer_sentiment'].apply(sentiment_category))

In [ ]:
#Create rated dataset
df_rated = df_clean.dropna(subset=['star_rating']).copy()

#Look at relationship
df_rated[['question_sentiment', 'answer_sentiment', 'star_rating']].corr()

,question_sentiment,answer_sentiment,star_rating
question_sentiment,1.000000,0.310488,0.165944
answer_sentiment,0.310488,1.000000,0.287288
star_rating,0.165944,0.287288,1.000000


#Save as a Parquet

In [ ]:
parquet_file = 'chat_metrics.parquet'
parquet_file

'chat_metrics.parquet'

In [ ]:
df_clean.to_parquet(parquet_file, index=False)

In [ ]:
!ls -l --si {parquet_file}

-rw-r--r-- 1 root root 302k Aug 17 17:27 chat_metrics.parquet


In [ ]:
df2 = pd.read_parquet(parquet_file)
df2.shape

(432, 42)

In [ ]:
os.environ["HF_TOKEN"] = userdata.get('hf_cs_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

'hf_ld ... iLe'

In [ ]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account

'stephanie465337'

In [ ]:
hf_org = "ddds-Capstone"
os.environ["HF_ORG"] = hf_org
hf_org

'ddds-Capstone'

In [ ]:
hf_repo = "Datasets"
os.environ["HF_REPO"] = hf_repo
hf_repo

'Datasets'

In [ ]:
!hf auth login --token $HF_TOKEN

Hint: A new version of huggingface_hub (1.27.0) is available! You are using version 1.23.0.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Capstone Token` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
%%capture hf_upload
%%bash
hf upload \
  --type dataset \
  ${HF_ORG}/${HF_REPO} \
  chat_metrics.parquet

In [ ]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/ddds-Capstone/Datasets/commit/450aab080ff757ffc1b0239da36bb14eaea6e9ce



In [ ]:
hf_url = f"https://huggingface.co/datasets/{hf_org}/{hf_repo}/resolve/main/chat_metrics.parquet"
hf_url

'https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/chat_metrics.parquet'